In [4]:
!pip install optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 8.6 MB/s eta 0:00:00


In [5]:
# Import necessary libraries
import optuna
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source


# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']


# Load the dataset
df = pd.read_csv(url, names=columns)
df.sample(5)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
331,2,87,58,16,52,32.7,0.166,25,0
646,1,167,74,17,144,23.4,0.447,33,1
61,8,133,72,0,0,32.9,0.270,39,1
317,3,182,74,0,0,30.5,0.345,29,1
670,6,165,68,26,168,33.6,0.631,49,0


In [6]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [7]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")


Training set shape: (537, 8)
Test set shape: (231, 8)


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
  # Suggest the values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )

  # Perform 3-fold cross-validation and calculte accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score

In [27]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler()) # We aim at maximizing the accuracy
study.optimize(objective, n_trials=50) # Run 50 trials to find the best hyperparameters

[I 2026-03-01 18:07:14,359] A new study created in memory with name: no-name-7931fb5f-7fad-44ba-9ff2-db18ff317257
[I 2026-03-01 18:07:16,147] Trial 0 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 182, 'max_depth': 9}. Best is trial 0 with value: 0.7616387337057727.
[I 2026-03-01 18:07:17,782] Trial 1 finished with value: 0.7746741154562383 and parameters: {'n_estimators': 133, 'max_depth': 15}. Best is trial 1 with value: 0.7746741154562383.
[I 2026-03-01 18:07:19,182] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 150, 'max_depth': 12}. Best is trial 1 with value: 0.7746741154562383.
[I 2026-03-01 18:07:20,284] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 160, 'max_depth': 5}. Best is trial 1 with value: 0.7746741154562383.
[I 2026-03-01 18:07:21,074] Trial 4 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 137, 'max_depth': 6}. Best is trial 1 with value: 0.7746741

In [10]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 133, 'max_depth': 18}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparamters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f"Test accuracy with best hyperparameters: {test_accuracy:.2f}")

Test accuracy with best hyperparameters: 0.74


## Samplers in Optuna

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators', 50,200)
  max_depth = trial.suggest_int('max_depth',3,20)

  # Create the RandomForestClassifier with suggested hyperparameters
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )

  # Perform 3-fold cross-validation and calculate accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return score # Return the accuracy score for Optuna to maximize

In [15]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler()) # We aim to maximize accuracy
study.optimize(objective, n_trials=50) # Run trials to find the best hyperparameters

[I 2026-03-01 17:43:41,613] A new study created in memory with name: no-name-10373ed7-0272-4d7d-81b6-190e1c58c3ed
[I 2026-03-01 17:43:43,570] Trial 0 finished with value: 0.756052141527002 and parameters: {'n_estimators': 183, 'max_depth': 6}. Best is trial 0 with value: 0.756052141527002.
[I 2026-03-01 17:43:45,216] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 165, 'max_depth': 18}. Best is trial 1 with value: 0.7690875232774674.
[I 2026-03-01 17:43:46,724] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 143, 'max_depth': 18}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-01 17:43:48,784] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 190, 'max_depth': 8}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-01 17:43:51,748] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 161, 'max_depth': 9}. Best is trial 2 with value: 0.772811918

In [16]:
# Print the best result
print(f"Best trial accuracy: {study.best_trial.value}")
print(f"Best Hyperparameters: {study.best_trial.params}")

Best trial accuracy: 0.7802607076350093
Best Hyperparameters: {'n_estimators': 121, 'max_depth': 18}


In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f"Test accuracy with best hyperparameters: {test_accuracy:.2f}")

Test accuracy with best hyperparameters: 0.74


In [19]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [20]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-03-01 17:57:18,644] A new study created in memory with name: no-name-b45c0ab9-aed1-41cc-8b35-622af35d2de1
[I 2026-03-01 17:57:19,596] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-01 17:57:21,327] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-03-01 17:57:21,820] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-01 17:57:22,934] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-03-01 17:57:23,557] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [21]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [22]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


## Optuna Visualizations

In [31]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [32]:
# 1. Optimizing History
plot_optimization_history(study).show()

In [34]:
# 2. Parallel coordinates plot
plot_parallel_coordinate(study).show()

In [35]:
# 3. Slice plot
plot_slice(study).show()

In [36]:
# 4. Contour Plot
plot_contour(study).show()

In [30]:
# 5. Hyperparamter Importance
plot_param_importances(study).show()

## Optimizing Multiple ML Models

In [37]:
# Importing Multiple ML models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [62]:
# Define the objective function for Optuna
def objective(trial):
  # Choose the algorithm to tune
  classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

  if classifier_name == 'SVM':
    # SVM Hyperparameters
    c = trial.suggest_float('C', 0.1, 100, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

  elif classifier_name == 'RandomForest':
    # Random Forest Hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        random_state=42
    )

  elif classifier_name == 'GradientBoosting':
    # Gradient Boosting Hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

  # Perform cross-validation and return the mean accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score

In [63]:
# Create a study and optimize it
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-01 19:58:19,936] A new study created in memory with name: no-name-c1dcff19-a8d3-46a6-a649-6913e9d18b90
[I 2026-03-01 19:58:28,015] Trial 0 finished with value: 0.7579143389199254 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 152, 'learning_rate': 0.014378076296455, 'max_depth': 18, 'min_samples_split': 3, 'min_samples_leaf': 4}. Best is trial 0 with value: 0.7579143389199254.
[I 2026-03-01 19:58:30,000] Trial 1 finished with value: 0.7653631284916201 and parameters: {'classifier': 'RandomForest', 'n_estimators': 125, 'max_depth': 20, 'min_samples_split': 9, 'min_samples_leaf': 3, 'bootstrap': False}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-03-01 19:58:32,604] Trial 2 finished with value: 0.7597765363128491 and parameters: {'classifier': 'RandomForest', 'n_estimators': 177, 'max_depth': 18, 'min_samples_split': 10, 'min_samples_leaf': 10, 'bootstrap': True}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-03-01 19:58:33,682] Trial 

In [64]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters: ", best_trial.params)
print("Best trial accuracy: ", best_trial.value)

Best trial parameters:  {'classifier': 'SVM', 'C': 0.15221107973462217, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy:  0.7895716945996275


In [65]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.757914,2026-03-01 19:58:19.942848,2026-03-01 19:58:28.015251,0 days 00:00:08.072403,NaN,NaN,GradientBoosting,NaN,NaN,0.014378,18.0,4.0,3.0,152.0,COMPLETE
1,1,0.765363,2026-03-01 19:58:28.018062,2026-03-01 19:58:30.000548,0 days 00:00:01.982486,NaN,False,RandomForest,NaN,NaN,NaN,20.0,3.0,9.0,125.0,COMPLETE
2,2,0.759777,2026-03-01 19:58:30.010740,2026-03-01 19:58:32.604498,0 days 00:00:02.593758,NaN,True,RandomForest,NaN,NaN,NaN,18.0,10.0,10.0,177.0,COMPLETE
3,3,0.780261,2026-03-01 19:58:32.607407,2026-03-01 19:58:33.682475,0 days 00:00:01.075068,NaN,False,RandomForest,NaN,NaN,NaN,6.0,2.0,2.0,99.0,COMPLETE
4,4,0.715084,2026-03-01 19:58:33.683888,2026-03-01 19:58:33.792574,0 days 00:00:00.108686,49.282163,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.711359,2026-03-01 19:59:18.118064,2026-03-01 19:59:18.150736,0 days 00:00:00.032672,0.131115,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-03-01 19:59:18.151898,2026-03-01 19:59:18.183841,0 days 00:00:00.031943,0.272796,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.785847,2026-03-01 19:59:18.185267,2026-03-01 19:59:18.218640,0 days 00:00:00.033373,0.201624,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.789572,2026-03-01 19:59:18.219830,2026-03-01 19:59:18.259045,0 days 00:00:00.039215,0.152457,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [66]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,79
RandomForest,11
GradientBoosting,10


In [67]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.745438
RandomForest,0.766040
SVM,0.775216


In [68]:
# 1. Optimization History
plot_optimization_history(study).show()

In [69]:
# 2. Slice plot
plot_slice(study).show()

In [70]:
# Hyperparameter Impotance
plot_param_importances(study).show()

In [11]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function
def objective(trial):
  # Hyperparameter search space
  param = {
    'verbosity': 0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss', # Ensure that the eval_metric is specified here
    'booster': 'gbtree',
    'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
    'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
    'eta': trial.suggest_float('eta', .01, 0.3),
    'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
    'max_depth': trial.suggest_int('max_depth', 3, 9),
    'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    'subsample': trial.suggest_float('subsample', 0.4, 1.0),
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
    'n_estimators': 300
  }

  # Create DMatrix for XGBoost
  dtrain = xgb.DMatrix(X_train, label=y_train)
  dtest = xgb.DMatrix(X_test, label=y_test)

  # Define a pruning callback based on evaluation metrics
  pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss") # Match the metric name in the evals list

  # Train the model
  bst = xgb.train(
      param,
      dtrain,
      num_boost_round=300,
      evals=[(dtrain, "train"), (dtest, "eval")], # Ensure the eval datasets and names are specified
      early_stopping_rounds=30,
      callbacks=[pruning_callback]
  )

  # Predict on the test set
  preds = bst.predict(dtest)
  best_preds = [int(np.argmax(line)) for line in preds]

  # Return accuracy as the objective value
  accuracy =  accuracy_score(y_test, best_preds)
  return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

[I 2026-03-02 05:11:12,208] A new study created in memory with name: no-name-b37561ea-4aa5-46b5-a13c-04cfd1c8441e


[0]	train-mlogloss:0.84112	eval-mlogloss:0.82496
[1]	train-mlogloss:0.66459	eval-mlogloss:0.63273
[2]	train-mlogloss:0.53586	eval-mlogloss:0.49834
[3]	train-mlogloss:0.43663	eval-mlogloss:0.39317
[4]	train-mlogloss:0.36935	eval-mlogloss:0.32289
[5]	train-mlogloss:0.31746	eval-mlogloss:0.26476
[6]	train-mlogloss:0.28379	eval-mlogloss:0.22807
[7]	train-mlogloss:0.27306	eval-mlogloss:0.21790
[8]	train-mlogloss:0.24724	eval-mlogloss:0.18864
[9]	train-mlogloss:0.23886	eval-mlogloss:0.17538
[10]	train-mlogloss:0.23139	eval-mlogloss:0.16667
[11]	train-mlogloss:0.23124	eval-mlogloss:0.16762
[12]	train-mlogloss:0.23007	eval-mlogloss:0.16660
[13]	train-mlogloss:0.22943	eval-mlogloss:0.16590
[14]	train-mlogloss:0.22921	eval-mlogloss:0.16659
[15]	train-mlogloss:0.22800	eval-mlogloss:0.16537
[16]	train-mlogloss:0.22732	eval-mlogloss:0.16285
[17]	train-mlogloss:0.22703	eval-mlogloss:0.16360
[18]	train-mlogloss:0.22725	eval-mlogloss:0.16368
[19]	train-mlogloss:0.22641	eval-mlogloss:0.16318
[20]	train

[I 2026-03-02 05:11:14,097] Trial 0 finished with value: 1.0 and parameters: {'lambda': 4.960338084001493e-07, 'alpha': 5.486571438628039e-05, 'eta': 0.2197450675095145, 'gamma': 6.85429596893445e-08, 'max_depth': 3, 'min_child_weight': 7, 'subsample': 0.6070679131016201, 'colsample_bytree': 0.9126435074335836}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.04651	eval-mlogloss:1.04747
[1]	train-mlogloss:0.97438	eval-mlogloss:0.97144
[2]	train-mlogloss:0.90844	eval-mlogloss:0.90118
[3]	train-mlogloss:0.84731	eval-mlogloss:0.83701
[4]	train-mlogloss:0.79341	eval-mlogloss:0.78143
[5]	train-mlogloss:0.74263	eval-mlogloss:0.72762
[6]	train-mlogloss:0.69720	eval-mlogloss:0.68150
[7]	train-mlogloss:0.65728	eval-mlogloss:0.63892
[8]	train-mlogloss:0.61978	eval-mlogloss:0.59856
[9]	train-mlogloss:0.58407	eval-mlogloss:0.56078
[10]	train-mlogloss:0.55211	eval-mlogloss:0.52641
[11]	train-mlogloss:0.52289	eval-mlogloss:0.49447
[12]	train-mlogloss:0.50597	eval-mlogloss:0.47790
[13]	train-mlogloss:0.47920	eval-mlogloss:0.44860
[14]	train-mlogloss:0.46456	eval-mlogloss:0.43472
[15]	train-mlogloss:0.44263	eval-mlogloss:0.41138
[16]	train-mlogloss:0.42273	eval-mlogloss:0.38945
[17]	train-mlogloss:0.40688	eval-mlogloss:0.37257
[18]	train-mlogloss:0.38730	eval-mlogloss:0.35198
[19]	train-mlogloss:0.37223	eval-mlogloss:0.33649
[20]	train

[I 2026-03-02 05:11:15,826] Trial 1 finished with value: 1.0 and parameters: {'lambda': 1.8433525831783102e-05, 'alpha': 7.810803851440684e-06, 'eta': 0.06065607654289753, 'gamma': 0.00717493689129004, 'max_depth': 5, 'min_child_weight': 5, 'subsample': 0.6625120936048907, 'colsample_bytree': 0.7382742343903381}. Best is trial 0 with value: 1.0.


...

[I 2026-03-02 05:11:29,476] Trial 47 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:0.92955	eval-mlogloss:0.93978


[I 2026-03-02 05:11:29,537] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:1.03056	eval-mlogloss:1.02750


[I 2026-03-02 05:11:29,568] Trial 49 pruned. Trial was pruned at iteration 1.


In [8]:

!pip install optuna-integration[xgboost]


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 1.2 MB/s eta 0:00:00


In [12]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()